In [1]:
import math
import mido
from mido import Message, MidiFile, MidiTrack
from pathlib import Path
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Helper to return an enumerated list without the element at a given index
def exclude(li, i):
    enum_list = list(enumerate(li))
    
    sublist = torch.cat([torch.tensor(enum_list[:i]), torch.tensor(enum_list[(i+1):])]).int().tolist()
    
    return sublist

In [3]:
# Train on contexts of multi-featured x-inputs to predict the label y of each feature
#
# classes - an array of class sizes. implies there are len(classes) classes/features.
# l1_size - the size of the linear layer.
# iters   - number of iterations to train
# x       - array of contexts
# y       - array of targets for each context

def train(classes, l1_size, iters, x, y):
    
    W1 = []    # Weight matrices for the linear layer
    b1 = []    # Biases for the linear layer
    W2 = []    # Weight matrices for the softmax layer
    b2 = []    # Biases for the softmax layer
    
    params = [W1, b1, W2, b2] # Used to compute loss
    
    probs = [None] * len(classes) # Feature -> [Examples x Feature range].
                                  # Index as [ Feature index, Example #, Label ] to get the probability of Label given Example for Feature
    loss  = []                    # Loss for each feature
    
    for _ in range(iters):
        for i in range(len(classes)):
            # forward pass
            if len(W1) == i:
                W1.append(torch.randn(X.shape[1] * X.shape[2], l1_size))
                W1[i].requires_grad = True
            if len(b1) == i:
                b1.append(torch.randn(l1_size))
                b1[i].requires_grad = True
        
            h1 = torch.tanh(x.float().view(x.shape[0], -1) @ W1[i] + b1[i])
            
            l2_size = classes[i]
            if len(W2) == i:
                W2.append(torch.randn(l1_size, l2_size))
                W2[i].requires_grad = True
            if len(b2) == i:
                b2.append(torch.randn(l2_size))
                b2[i].requires_grad = True
        
            h2 = h1 @ W2[i] + b2[i]
            
            counts = h2.exp()
            probs[i] = (counts / counts.sum(1, keepdims=True)).view(-1, l2_size)
            
            _loss = -probs[i][torch.arange(x.shape[0]), y.int() @ F.one_hot(torch.tensor(i), num_classes=len(classes)).int()].log().mean()
            if len(loss) == i:
                loss.append(_loss)
            else:
                loss[i] = _loss
        
            # backward pass
            for p in params:
                p[i].grad = None
            loss[i].backward()
        
            # update
            for p in params:
                p[i].data += -0.1 * p[i].grad

    return [W1, b1, W2, b2, classes, loss]
                
def forward(model, x):
    W1 = model[0]
    b1 = model[1]
    W2 = model[2]
    b2 = model[3]
    classes = model[4]
    params = [W1, b1, W2, b2]
    
    probs = []
    for i in range(len(classes)):
        # forward pass
        h1 = torch.tanh(x.float().view(x.shape[0], -1) @ W1[i] + b1[i])
        h2 = h1 @ W2[i] + b2[i]
        
        counts = h2.exp()
        if len(probs) == i:
            probs.append([])
        probs[i] = (counts / counts.sum(1, keepdims=True)).view(-1, classes[i])

    return probs

In [4]:
CH_NOTE1 = 0
CH_NOTE2 = 1
CH_NOTE3 = 2
CH_NOTE4 = 3
CH_NOTE5 = 4
CH_NOTE6 = 5
CH_NOTE7 = 6
CH_DURATION = 7

In [5]:
def build_chords_dataset(path):    
    chords = []
    chord = [0] * 8
    last_chord = None
    notes = []
    note_idx = 0
    
    for path in Path(path).iterdir():
        mid = mido.MidiFile(path)
        
        for msg in mid.tracks[0]:
            if not msg.is_meta and not msg.type == 'sysex':
                if msg.time != 0:
                    chords.append(chord)
                    last_chord = chord
                    last_chord[CH_DURATION] = msg.time
                    chord = [0] * 8
                    note_idx = 0
                if msg.type == 'note_on':
                    if msg.time != 0:
                        chord[note_idx] = msg.note
                        note_idx += 1
                    chord[note_idx] = msg.note
                    note_idx += 1
                    notes.append(msg.note)
                else:
                    notes.remove(msg.note)
    
    context_length = 4
    context = [[0] * 8] * context_length
    X, Y = [], []
    X.append(context)
    Y.append(chords[0])
    for c1, c2 in zip(chords, chords[1:]):
        context = context[1:] + [c1]
        X.append(context)
        Y.append(c2)
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

In [6]:
def generate_chords(chords_model, bars):
    conv_xfrm_exp = 1
    diff = 0
    max_diff = 9
    out = []
    context = [[0] * 8] * context_length

    while torch.tensor([c[7] for c in out]).sum() < bars * 1920:
        boost = 1

        # Generate a message until it is valid
        while True:
            msg = [0] * 8
            for i in range(8):
                x = torch.tensor([context])
                probs = forward(chords_model, x)
                pprobs = probs[i] if i < CH_NOTE5 or i == CH_DURATION else probs[torch.rand(1).int().item() * 4]
                if diff > max_diff:
                    pprobs_xfrm  = pprobs**conv_xfrm_exp
                    pprobs_xfrm = pprobs_xfrm / pprobs_xfrm.sum()
                    pprobs = pprobs_xfrm
                while True:
                    ix = torch.multinomial(pprobs, num_samples=1).item()
                    if i == CH_NOTE1 or i == CH_DURATION:
                        break
                    diffs = [abs(ix - o) for o in msg[:i]] # Compute differences of sample from every stored sample up to i
                    if set(diffs).isdisjoint(set(range(2))):
                        break
                
                msg[i] = ix
            msg_avg = torch.tensor(msg[:CH_DURATION]).sum() / 7
            ctx_avg = torch.tensor(context[-1][:CH_DURATION]).sum() / 7
            diff = abs(msg_avg - ctx_avg)

            # if the context is full and diff > max_diff
            if True not in [torch.tensor(m).sum() == 0 for m in context] and diff > max_diff:
                conv_xfrm_exp += 0.5
                print(f'(convex transform applied to probabilities because abs(msg_avg - ctx_avg) = {diff})')
                continue
            break
    
        msg[CH_DURATION] = ((torch.rand(1)*2).int().item() * 2 + 2) * 480
        context = context[1:] + [msg]
        out.append(msg)

    # Trim to the end of the last bar
    deltas = torch.tensor([c[CH_DURATION] for c in out])
    length = deltas.sum()
    rem = length % 1920
    rem = rem.item()

    out[-1][CH_DURATION] -= rem
    
    return out

In [7]:
# Each chord has incompatible notes removed.
# A note is removed if it is too close to another note.
# If the note is farther than two octaves (24 intervals) from any other, a separate note is removed if it is the farthest from the others, that is, if it is the outlier
def clean_chords(progression):
    new_prog = []
    for chord in progression:
        candidates = list(range(7))
        to_remove = []
        
        while True:
            chord_compat = [ [ abs(chord[i] - j) if set(to_remove).isdisjoint([ji]) else math.inf for ji, j in exclude(chord[:7], i) ] for i in range(7) ]
            
            removed_one = False
        
            for i,_ in enumerate(chord_compat):
                if i in to_remove:
                    continue
                i_too_close = False
                for nn in chord_compat[i]:
                    if (nn in range(3)):
                        to_remove.append(i)
                        i_too_close = True
                        break
    
                chord_compat_noinf = [val for val in chord_compat[i] if not val == math.inf]
                if not len(chord_compat_noinf) == 0:
                    i_too_far = False
                    if not i_too_close and torch.tensor(chord_compat_noinf).max() > 24:
                        # index to remove
                        # ---------------
                        # farthest away impl
                        # idx = chord_compat[i].index(torch.tensor(chord_compat[i]).max())
                        # if idx >= i:
                        #     idx += 1
                        # to_remove.append(idx)
                        
                        # outlier impl
                        note_diffs = [ [abs(a - b) for b in [v for _, v in exclude(chord[:CH_DURATION], a)]] for a in chord[:CH_DURATION] ]
                        note_diffs_sums = [ torch.tensor([e for e in row if not e == math.inf]).sum() for row in note_diffs ]
                        idx2 = note_diffs_sums.index(torch.tensor([v if not i in to_remove else -math.inf for i, v in enumerate(note_diffs_sums) ]).max())
                        to_remove.append(idx2)
                        i_too_far = True
    
                if i_too_close or i_too_far:
                    removed_one = True
                    break
    
            if not removed_one:
                break
            
        pchord = [val for i, val in enumerate(chord) if not i in to_remove and not i == CH_DURATION]
        end = torch.zeros(8 - len(pchord)).tolist()
        end[-1] = chord[CH_DURATION]
        pchord += end
        
        new_prog.append(pchord)

    return new_prog

In [8]:
PH_DIRECTION  = 0 # Internal note direction
PH_LENGTH     = 1 # Length in notes
PH_DURATION   = 2 # Phrase duration
PH_FIRST_NOTE = 3 # First note
PH_HIST_DIR   = 4 # First note relationship with first note phrase_hist phrases before

In [9]:
def build_phrases_dataset(path):
    out = []
    
    for path in Path(path).iterdir():
        mid = mido.MidiFile(path)
    
        last_msg = None
    
        direction = None
        count = 0
        duration = 0
    
        first_note = None

        phrase = torch.zeros(5).tolist()
    
        melody = []
        
        for msg in mid.tracks[0]:
            if not msg.is_meta and not msg.type == 'sysex':
                if msg.type == 'note_on':
                    if last_msg is not None:
                        if first_note is None:
                            first_note = msg.note
                        diff = msg.note - last_msg.note
                        _direction = 0 if diff == 0 else diff / abs(diff)
        
                        if direction is not None and _direction != direction:
                            phrase[PH_DIRECTION] = direction
                            phrase[PH_LENGTH] = count
                            phrase[PH_DURATION] = duration
                            phrase[PH_FIRST_NOTE] = first_note
                            
                            diff = 0 if len(melody) < 2 else first_note - melody[-phrase_hist][3]
                            phrase[PH_HIST_DIR] = 0 if diff == 0 else diff / abs(diff)
                            
                            melody.append(phrase)
                            
                            count = 0
                            duration = 0
                            direction = None
                            first_note = None
                            
                            phrase = torch.zeros(5).tolist()
                        elif direction is None:
                            direction = _direction
                        
                    last_msg = msg
                else:
                    if msg.time != 0:
                        count += 1
                        duration += msg.time
        out.append(melody)
    
    context_length = 4
    context = [[0] * 5] * context_length
    X, Y = [], []
    for m in out:
        context = [[0] * 5] * context_length
        for c1, c2 in zip(m, m[1:]):
            context = context[1:] + [c1]
            X.append(context)
            Y.append(c2)
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

In [10]:
def generate_phrases(phrase_model, bars):
    out = []
    context = [[0] * 5] * context_length

    while torch.tensor([p[PH_DURATION] for p in out]).sum() < bars * 1920:
        msg = [0] * 5
        for i in range(4):
            x = torch.tensor([context])
            probs = forward(phrase_model, x)
            pprobs = probs[i]
            ix = torch.multinomial(pprobs, num_samples=1).item()

            if i == 0:
                ix -= 1
            
            msg[i] = ix

        diff = 0 if len(out) < phrase_hist else msg[PH_FIRST_NOTE] - out[-phrase_hist][PH_FIRST_NOTE]
        msg[PH_HIST_DIR] = 0 if diff == 0 else diff / abs(diff)
        context = context[1:] + [msg]
        out.append(msg)

    rem = torch.tensor([p[PH_DURATION] for p in out]).sum() % 1920
    out[-1][PH_DURATION] -= rem.item()
    
    return out

In [11]:
def phrases_by_chord(progression, phrases):
    phrases_copy = phrases.copy()
    
    chords_phrases = []
    phrase_i = 0
    
    for chord in progression:
        chord_phrases = []
        chord_time = 0
        rem = 0
        while True:
            ph = phrases_copy[phrase_i]
            chord_phrases.append(ph)
            chord_time += ph[PH_DURATION] + rem
            phrase_i += 1

            # If this note went past the end of the chord, remove the remainder and put the remainder back into the time of the next phrase. No time is lost.
            if chord_time >= chord[CH_DURATION]:
                rem = chord_time - chord[CH_DURATION]
                chord_phrases[-1][PH_DURATION] -= rem
                if phrase_i < len(phrases_copy):
                    phrases_copy[phrase_i-1][PH_DURATION] -= rem
                    phrases_copy[phrase_i][PH_DURATION] += rem
                break
        chords_phrases.append([chord, chord_phrases])

    return chords_phrases

In [12]:
def build_times_dataset(path):
    out = []
    
    for path in Path(path).iterdir():
        mid = mido.MidiFile(path)
        phrase = torch.zeros(5).tolist()
    
        melody = []
        time = 0
        skip = True
        
        for msg in mid.tracks[0]:
            if not msg.is_meta and not msg.type == 'sysex':
                if msg.type == 'note_on':
                    if skip:
                        skip = False
                        continue
                    if msg.time != 0:
                        melody.append(msg.time)
                if msg.type == 'note_off':
                    if msg.time != 0:
                        melody.append(msg.time)
                        skip = True
                        
        out.append(melody)
    
    context_length = 4
    context = [[0]] * context_length
    X, Y = [], []
    for timeset in out:
        context = [[0]] * context_length
        for c1, c2 in zip(timeset, timeset[1:]):
            context = context[1:] + [[c1]]
            X.append(context)
            Y.append([c2])
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

In [13]:
def generate_times(times_model, bars):
    boost = 1
    out = []
    context = [0] * context_length

    while torch.tensor(out).sum() < bars * 1920:
        x = torch.tensor([context])
        probs = forward(times_model, x)
        pprobs = probs[0]
        ix = max(120, ((torch.multinomial(pprobs, num_samples=1) / 120).int() * 120).item())

        context = context[1:] + [ix]
        out.append(ix)

    rem = (torch.tensor(out).sum() % 1920).item()
    out[-1] -= rem
    
    return out

In [14]:
NT_NAME     = 0
NT_DURATION = 1

In [15]:
def generate_notes(chords_phrases, times):
    times_copy = times.copy()
    notes = []
    last_i = -1
    for i, (chord, phrases) in enumerate(chords_phrases):
        if i > last_i:
            chord_time = 0
            last_i = i


        # While there is time to fill in the duration of chord, take durations from times to assign to the notes.
        # The first note for the chord is the starting note of the first phrase.
        # To choose the name of further notes, move in the current phrase's direction until a name is found in the chord
        name = chord[CH_NOTE1]
        while chord_time < chord[CH_DURATION]:
            for phrase in phrases:
                phrase_count = 0     
                direction = phrase[PH_DIRECTION]
                while phrase_count < phrase[PH_LENGTH]:
                    if direction == 0:
                        direction = [-1, 1][(torch.rand(1) * 2).int().item()]
                    name += direction
                    while name not in chord[:CH_NOTE5]:
                        if name > torch.tensor(chord[:CH_NOTE5]).max():
                            direction = -1
                        if name < torch.tensor(chord[:CH_NOTE5]).min():
                            direction = 1
                        name += direction

                    # If the duration for this note goes past the end of the chord,
                    # take the remaining time out of the duration and put it into the next element of times
                    if chord_time + times_copy[0] > chord[CH_DURATION]:
                        rem = chord_time + times_copy[0] - chord[CH_DURATION]
                        if len(times_copy) == 0:
                            times_copy = [rem]
                        else:
                            times_copy[0] -= rem
                            if len(times_copy) > 1:
                                times_copy[1] += rem
                        
                    note = [name, times_copy[0]]
                    chord_time += times_copy[0]
                    
                    times_copy = times_copy[1:]
        
                    notes.append(note)
                    phrase_count += 1
                    
                    if chord_time >= chord[CH_DURATION]:
                        break 
                if chord_time >= chord[CH_DURATION]:
                    break

    return notes

In [16]:
# Put it all together

In [17]:
chords_path = '/Users/nir/MIDI/Unison MIDI Chord Pack/Unison MIDI Chord Pack/01 - C Major - A Minor/4 Progressions/1 Diatonic Triads/Major Progressions/'
mldies_path = '/Users/nir/MIDI/Unison Essential MIDI Melodies/'

l1_size        = 100  # Size of the linear layer
tr_iters       = 1000 # Training iterations
context_length = 4    # Training context length
phrase_hist    = 2    # Phrase history length

print('Building chords dataset...')
X, Y = build_chords_dataset(chords_path)

print(f'Training chords model...')
classes = [88, 88, 88, 88, 88, 88, 88, 5000]
chords_model = train(classes, l1_size, tr_iters, X, Y)

print('Building phrases dataset...')
X, Y = build_phrases_dataset(mldies_path)

print(f'Training phrases model...')
classes = [3, 8, 1000, 88, 3]
phrase_model = train(classes, l1_size, tr_iters, X, Y)

print('Building durations dataset...')
X, Y = build_times_dataset(mldies_path)

print(f'Training durations model...')
classes = [1000]
times_model = train(classes, l1_size, tr_iters, X, Y)

Building chords dataset...
Training chords model...
Building phrases dataset...
Training phrases model...
Building durations dataset...
Training durations model...


In [18]:
def save_midi(chords, melodies, path, mtype=1):
    if mtype == 0:
        tracks = []
        
        track = MidiTrack()
        running_time = 0
        for chord in chords:
            for n in range(min(4, len(chord))):
                if chord[n] != 0:
                    track.append(Message('note_on', note=chord[n], velocity=100, channel=1, time=0))
                    
            for n in range(min(4, len(chord))):
                if chord[n] != 0:
                    track.append(Message('note_off', note=chord[n], velocity=100, channel=1, time=chord[CH_DURATION] if n == 0 else 0))
        tracks.append(track)
            
        for mi, melody in enumerate(melodies):
            track = MidiTrack()
            tracks.append(track)
                
            for note in melody:
                track.append(Message('note_on', note=note[NT_NAME], velocity=100, channel=2+mi, time=0))
                track.append(Message('note_off', note=note[NT_NAME], velocity=100, channel=2+mi, time=note[NT_DURATION]))

        merged_track = mido.merge_tracks(tracks)
        
        # 3. Create a new MidiFile with type=0
        type0_mid = mido.MidiFile(type=0)
        type0_mid.tracks.append(merged_track)
        
        # 4. Save the file
        type0_mid.save(path)

    else:
        mid = MidiFile()
        track = MidiTrack()
        mid.tracks.append(track)
        
        for chord in chords:
            for n in range(min(4, len(chord))):
                if chord[n] != 0:
                    track.append(Message('note_on', note=chord[n], velocity=100, channel=1, time=0))
            for n in range(min(4, len(chord))):
                if chord[n] != 0:
                    track.append(Message('note_off', note=chord[n], velocity=100, channel=1, time=chord[CH_DURATION] if n == 0 else 0))

        for melody in melodies:
            track = MidiTrack()
            mid.tracks.append(track)
            
            for note in melody:
                track.append(Message('note_on', note=note[NT_NAME], velocity=100, channel=1, time=0))
                track.append(Message('note_off', note=note[NT_NAME], velocity=100, channel=1, time=note[NT_DURATION]))
                    
        mid.save(path)
    print(f'Saved {path}.')

In [19]:
num_bars = 16 # Number of bars to generate

print(f'Generating chords...')
chords = clean_chords(generate_chords(chords_model, num_bars))

print(f'Generating phrases...')
phrases = generate_phrases(phrase_model, num_bars)

print(f'Generating note times...')
times = generate_times(times_model, num_bars)

print(f'Generating notes...')
chords_phrases = phrases_by_chord(chords, phrases)
notes = generate_notes(chords_phrases, times)

save_midi(chords, [notes], 'song.mid')

Generating chords...
Generating phrases...
Generating note times...
Generating notes...
Saved song.mid.
